In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, mean_squared_error, confusion_matrix, classification_report

## Load All Data

In [3]:
from data_utils import get_images, get_labels
disaster_list = ["midwest-flooding", "socal-fire"]
data = {}
split = "train"
with open('config.json') as config_file:
    config = json.load(config_file)
    data_dir = config['data_dir']

for disaster in disaster_list:
    print(f"Loading {split} images and labels for {disaster} dataset...")
    images = get_images(data_dir, disaster, split=split)
    labels = get_labels(data_dir, disaster, split=split)
    data[disaster] = {"images": images, "labels": labels}

Loading train images and labels for midwest-flooding dataset...
Loading train images and labels for socal-fire dataset...


# Binary Classification: Flood vs. Fire

## Process Data

In [4]:
images = np.array(data['midwest-flooding']['images'] + data['socal-fire']['images'])
labels = np.array([0] * len(data['midwest-flooding']['labels']) + [1] * len(data['socal-fire']['labels']))

# Normalize and extract RGB means (N x H x W x 3 --> N x 3)
images_normalized = images / 255.0
rgb_means = np.mean(images_normalized, axis=(1, 2))

X, Y = rgb_means, labels
X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size=0.2, random_state=42)

In [5]:
def straified_CV(model, n_split=5, metric=accuracy_score):
    skf = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=42)

    values = []
    for train_index, test_index in skf.split(X, Y):
        X_train, X_test = X[train_index], X[test_index]
        Y_train, Y_test = Y[train_index], Y[test_index]
        
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)
        
        values.append(metric(Y_test, Y_pred))

    return np.mean(values)

## Logistic Regression Model

In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42).fit(X_train, Y_train)

train_preds = model.predict(X_train)
valid_preds = model.predict(X_valid)

print("Training Accuracy:  ", accuracy_score(train_preds, Y_train))
print("Validation Accuracy:", accuracy_score(valid_preds, Y_valid))

Training Accuracy:   0.7406103286384976
Validation Accuracy: 0.7336448598130841


In [7]:
print(straified_CV(model, metric=accuracy_score))

0.739212847176517


## Support Vector Machine

In [8]:
from sklearn.svm import SVC

model = SVC(kernel='poly', C=1.0).fit(X_train, Y_train)

train_preds = model.predict(X_train)
valid_preds = model.predict(X_valid)

print("Training Accuracy:  ", accuracy_score(train_preds, Y_train))
print("Validation Accuracy:", accuracy_score(valid_preds, Y_valid))

Training Accuracy:   0.9647887323943662
Validation Accuracy: 0.9579439252336449


In [9]:
print(straified_CV(model, metric=accuracy_score))

0.9624851915229694


## Boosted Trees

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=100,  # number of boosting rounds
    learning_rate=0.1, # step size
    max_depth=3,       # tree depth
    random_state=42
).fit(X_train, Y_train)

train_preds = model.predict(X_train)
valid_preds = model.predict(X_valid)

print("Training Accuracy:  ", accuracy_score(train_preds, Y_train))
print("Validation Accuracy:", accuracy_score(valid_preds, Y_valid))

Training Accuracy:   0.994131455399061
Validation Accuracy: 0.9205607476635514


In [11]:
print(straified_CV(model, metric=accuracy_score))

0.9409284366635953
